# Re-ranking Strategies for RAG on NFCorpus

Diversity-aware (Aspect-Aware) and cross-encoder re-rankers, compared with BM25 and dense retrieval.

This notebook walks through the three experiments using the modules in `src/`.
Install the dependencies first (`pip install -r requirements.txt`) and open the notebook from the `notebooks/` folder.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from src import config
from src.aspects import extract_aspects
from src.pipeline import build_resources, rank_all_systems, set_seed

set_seed(config.SEED)

## 2. Data, models and indexes

Downloads NFCorpus (BEIR) on first use, encodes the 3,633 documents once and builds the BM25 and FAISS indexes.

In [ ]:
res = build_resources()

## 3. Aspect extraction

In [ ]:
for q in ["Do Cholesterol Statin Drugs Cause Breast Cancer?",
          "What are the effects of COVID-19 on the heart?",
          "Exploiting Autophagy to Live Longer"]:
    print(f"{q}\n  -> {extract_aspects(q)}")

## 4. One query through every system

Both re-rankers receive the same top-20 candidates from each first-stage retriever.

In [ ]:
query = next(iter(res.queries.values()))
rankings, aspects, _ = rank_all_systems(res, query)
print("Query  :", query)
print("Aspects:", aspects)
for system, ranked in rankings.items():
    print(f"\n{system}")
    for rank, i in enumerate(ranked, start=1):
        print(f"  {rank}. {res.doc_texts[i][:90]}...")

## 5. Experiment 1: retrieval metrics and Aspect Coverage Score

In [ ]:
from src.experiments import run_retrieval_eval
from src.plots import plot_retrieval

summary, per_query, significance = run_retrieval_eval(res)
summary.round(4)

In [ ]:
significance.round(4)

In [ ]:
plot_retrieval(summary, config.FIGURES_DIR / "retrieval_comparison.png")
display(Image(str(config.FIGURES_DIR / "retrieval_comparison.png")))

## 6. Experiment 2: λ sensitivity of the Aspect-Aware re-ranker

In [ ]:
from src.experiments import run_lambda_sweep
from src.plots import plot_lambda

sweep = run_lambda_sweep(res)
sweep.round(4)

In [ ]:
plot_lambda(sweep, config.FIGURES_DIR / "lambda_sensitivity.png")
display(Image(str(config.FIGURES_DIR / "lambda_sensitivity.png")))

## 7. Experiment 3: end-to-end RAG with flan-t5-base (ROUGE-L)

In [ ]:
from src.data import load_rag_qa
from src.experiments import run_rag_eval
from src.generation import AnswerGenerator
from src.plots import plot_rag

scores, answers, retrieved = run_rag_eval(res, AnswerGenerator(), load_rag_qa())
scores.round(4)

In [ ]:
scores.drop(columns="Query").mean().round(4)

In [ ]:
plot_rag(scores, config.FIGURES_DIR / "rag_comparison.png")
display(Image(str(config.FIGURES_DIR / "rag_comparison.png")))

## 8. Save results

In [ ]:
config.RESULTS_DIR.mkdir(exist_ok=True)
summary.round(4).to_csv(config.RESULTS_DIR / "reranking_results.csv", index=False)
sweep.round(4).to_csv(config.RESULTS_DIR / "lambda_analysis.csv", index=False)
scores.round(4).to_csv(config.RESULTS_DIR / "rag_comparison.csv", index=False)
res.save_ce_cache()